# 06 — Generation, Grounding, Citations & LCEL Chains

**By the end of this notebook, you can assemble clean LangChain LCEL (`|`) pipelines that cite sources strictly from context and honestly say 'I don't know' on gaps.**

### Learning Objectives
- Design defensive RAG system prompts that constrain answers strictly to retrieved evidence.
- Build composable pipelines using LangChain Expression Language (`RunnablePassthrough`, `ChatPromptTemplate`, `StrOutputParser`).
- Verify grounded answers include source citations and refuse unanswerable questions without fabricating facts.

### Prerequisites
- Notebooks 00–05

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Grounded Prompting & LCEL Pipes

Think of **LCEL (LangChain Expression Language)** like Unix command-line pipes (`|`) or a factory assembly line: output from step 1 flows cleanly into step 2!

```
question ---> { "context": retriever | format_docs, "question": pass_through } ---> prompt ---> llm ---> clean string
```

Two rules make a production prompt trustworthy:
1. **Strict Grounding Guardrail**: Explicitly tell the LLM to ignore outside training memory and answer *only* from `<context>`.
2. **Explicit Refusal Clause**: Instruct the LLM: *"If the answer is not in the context, reply: I don't know based on the provided context."*

```
User Question --+--> Retriever --> format_docs (adds [source] tags) --+
                |                                                     v
                +--> RunnablePassthrough() -------------------> ChatPromptTemplate --> LLM --> Cited String
```

## 3. Step-by-Step Build

### Step 1: Prepare Vector Store & Format-Docs Helper with Source Labels
Let's create our vector store and a tiny `format_docs` function that prepends `[filename]` onto every retrieved chunk so citations stay attached.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=360, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
retriever = InMemoryVectorStore.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source', 'unknown')}]\n{d.page_content}" for d in docs)
        

> **What just happened?**
> Now every text block inside `<context>` visibly carries its origin filename.

### Step 2: Craft a Strict Grounded Prompt Template
We wrap retrieved context cleanly inside `<context>...</context>` XML delimiters so instructions never blur together with source text.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
rag_prompt = ChatPromptTemplate.from_template(
    "You are a concise, truthful assistant. Answer using ONLY the context below.\n"
    "If the context lacks the answer, say 'I don't know based on the provided context.'\n\n"
    "<context>\n{context}\n</context>\n\nQuestion: {question}"
)
print("Input variables expected:", rag_prompt.input_variables)
        

Input variables expected: ['context', 'question']


> **What just happened?**
> `ChatPromptTemplate` validates variable names automatically (`['context', 'question']`).

### Step 3: Compose the End-to-End LCEL Chain with `|`
Let's snap our retriever, formatter, prompt, LLM, and string output parser together into a single runnable `rag_chain`.

In [4]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("LCEL chain ready!")
        

LCEL chain ready!


> **What just happened?**
> Calling `rag_chain.invoke('...')` executes retrieval, formatting, prompt templating, generation, and string parsing in one shot.

### Step 4: Test Both Answerable Questions and Unanswerable Refusals
Let's test `rag_chain` on a specific factual question AND on a missing-fact question (Novasphere's NASDAQ stock ticker).

In [5]:
q_ans = "When will legacy REST API v1 be shut down completely, and what replaces it?"
q_unans = "What is Novasphere Dynamics' stock ticker symbol on NASDAQ?"
print("Answerable   :", rag_chain.invoke(q_ans))
print("Unanswerable :", rag_chain.invoke(q_unans))
        

Answerable   : Based on the provided context: ### Important API Deprecation Notice **Legacy REST API v1** is officially scheduled for complete retirement. After **December 31, 2026**, all REST API v1 endpoints will permanently return **HTTP 410 Gone**. All engineering teams and customers must migrate integrations to **gRPC v2** or the **HyperSync Streaming API** prior to the shutdown date.
Unanswerable : I don't know based on the provided context. The supplied documents do not contain enough information to answer this question.


> **What just happened?**
> Notice how it answers December 31, 2026 / gRPC v2 accurately for the real question, and cleanly refuses to guess a fake ticker symbol!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Return both answer AND source documents** — *Hint:* Use `RunnableParallel` or inspect `retriever.invoke(q)` alongside `rag_chain.invoke(q)`.
2. **Ask about Zurich pet days** — *Hint:* Verify the chain cites `Tuesdays and Thursdays`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
q_pets = "On which days are pets allowed in the Zurich office hub?"
used_docs = retriever.invoke(q_pets)
print("Answer :", rag_chain.invoke(q_pets))
print("Sources:", sorted({d.metadata['source'] for d in used_docs}))
        

Answer :

 Based on the provided context: Zurich Hub Pet Policy: Well-behaved dogs are permitted in the Zurich hub on Tuesdays and Thursdays only, provided they are registered with office facilities and have valid vaccinations on file. Pets are not permitted in robotics hardware testing bays.
Sources: ['company_handbook.txt', 'policy_brief.pdf']


## 5. Common Mistakes
- Forgetting `<context>` delimiters around retrieved snippets — makes it harder for LLMs to separate rules from raw document text.
- Omitting explicit 'say I don't know' instructions, tempting helpful models to hallucinate guesses.
- Returning plain answer strings in production without tracking the source filenames.

## 6. Recap
- Grounded prompting pairs `<context>` isolation with explicit refusal rules.
- LCEL (`|`) chains compose retriever, formatter, prompt, LLM, and parser cleanly.
- Testing BOTH answerable and unanswerable queries early proves your guardrails actually work.

## 7. Next Notebook
Next up (`07_query_translation.ipynb`): What happens when users write vague or multi-part questions? Learn Multi-Query, RAG-Fusion, Decomposition, Step-Back, and HyDE!

## 8. Video Chapter Outline
1. **Designing Truthful Prompts** — XML delimiters + refusal rules.
2. **LCEL Assembly Line** — How `RunnablePassthrough` and pipes work.
3. **Source Attaching with `format_docs`** — Making citations natural and auditable.
4. **Live Answer vs Refusal Tests** — Verifying groundedness on NASDAQ gap question.
5. **Wrap-up** — End of Part 2 — ready for Advanced RAG!